# IBM Watson for Oncology: When "Trained on Data" Means One Hospital's Opinions

In this notebook I look at IBM Watson for Oncology as a tenth mechanism behind an AI ethics failure: a system marketed as having learned from vast medical evidence that had actually learned from a small number of hypothetical cases written by oncologists at a single hospital, then was deployed globally as if their judgment generalized everywhere.

In this notebook, I will:

- Summarize what Watson for Oncology was built on, and what investigations found once it was deployed abroad
- Simulate a narrow, single-institution training set standing in for Watson's actual "ground truth" cases
- Measure two different kinds of failure this produces once the resulting recommendations are deployed to a broader, different population: an unsafe contraindication, and a recommendation nobody there can actually act on
- Try the fix that worked in an earlier notebook, broadening the training data, and see it only closes one of the two gaps
- Find the fix that closes the other: a check that has nothing to do with training data at all

## 1. Background: What Was Reported

IBM Watson for Oncology, developed starting around 2012 in partnership with Memorial Sloan Kettering Cancer Center (MSK), was marketed as an AI system able to recommend cancer treatments by synthesizing vast quantities of medical literature and patient data. A 2018 investigation by STAT News, along with internal IBM training materials that later became public, found that in practice the system's treatment recommendations were built primarily from a relatively small number of hypothetical patient cases, written and annotated by a handful of oncologists at MSK to reflect MSK's own treatment preferences, not from machine learning over large, real-world, multi-institution patient outcome data as many health systems buying the product understood it to be.

When the system was deployed to hospitals in other countries, including India and South Korea, with different patient populations, different national treatment guidelines, and different drug approvals and formularies than MSK's, it sometimes recommended treatments that were unsafe for a given patient's specific risk factors, or simply unavailable in that country at all. Internal IBM documents reportedly described multiple examples of "unsafe and incorrect" treatment recommendations. Independent studies at hospitals that adopted the system found its recommendations agreed with local oncologists at very different rates depending on cancer type, matching closely for some cancers with well-standardized, widely agreed treatment guidelines, and matching far less often for others where legitimate practice varies more between institutions. Several hospitals, including a well-known cancer center in Denmark, ended their contracts after concluding the system's recommendations weren't reliable enough for their patient population.

## 2. Why This Is a Different Mechanism

This isn't the facial recognition notebook's underrepresentation problem, where a group was a small share of otherwise real, broad training data. Here the entire foundation the system was said to be built on, a large body of real-world medical evidence, was substantially replaced by something narrower: a curated set of hypothetical scenarios reflecting one hospital's specific judgment calls, treated as though it were generalizable ground truth.

And once deployed, the resulting recommendations ran into two genuinely different problems at once: a recommendation could be clinically wrong for a specific patient because the narrow training data never adequately covered their situation, or a recommendation could be perfectly reasonable in the abstract and still be useless, because the drug or approach it named wasn't something the deploying hospital could actually give.

## 3. A Note on the Simulation Below

This notebook does not use Watson for Oncology's actual case library or its real recommendation logic, neither of which was ever fully published. I represent MSK's synthetic training cases and a broader deployment population with a few simplified patient attributes, to make two specific, separate mechanisms concrete: a narrow training set producing an unsafe blind spot, and a correct-looking recommendation running into a local constraint no amount of training data would fix.

## 4. Simulating a Broad Deployment Population and a Narrow Training Set

I generate a broad population of patients across several deployment hospitals, a small fraction of whom have a bleeding-risk condition that makes the standard drug dangerous for them. Separately, I generate MSK's own small set of hypothetical training cases, where that same bleeding-risk condition is far rarer, mirroring how a handful of oncologists writing hypothetical cases at one hospital would tend to reflect that hospital's own typical patient mix rather than the full range of patients anywhere the product might later be sold.

In [ ]:
import random

random.seed(11)

DEPLOYMENT_BLEEDING_RISK_RATE = 0.08
TRAINING_BLEEDING_RISK_RATE = 0.02
N_DEPLOYMENT_PATIENTS = 5000
N_TRAINING_CASES = 120

HOSPITALS = [
    {"name": "MSK-based reference hospital", "drug_available": True},
    {"name": "Partner hospital, country A", "drug_available": True},
    {"name": "Partner hospital, country B", "drug_available": False},
    {"name": "Partner hospital, country C", "drug_available": False},
]


def make_patients(n, bleeding_risk_rate, hospitals):
    """Returns a list of patient dicts with a bleeding-risk flag and an assigned hospital."""
    patients = []
    for _ in range(n):
        hospital = random.choice(hospitals)
        patients.append({
            "has_bleeding_risk": random.random() < bleeding_risk_rate,
            "hospital": hospital["name"],
            "drug_available": hospital["drug_available"],
        })
    return patients


deployment_patients = make_patients(N_DEPLOYMENT_PATIENTS, DEPLOYMENT_BLEEDING_RISK_RATE, HOSPITALS)
training_cases = make_patients(N_TRAINING_CASES, TRAINING_BLEEDING_RISK_RATE, [HOSPITALS[0]])

## 5. Deriving Watson's Recommendation Rule and Checking for a Safety Blind Spot

A recommendation system built from only 120 hypothetical cases, very few of which involve bleeding risk, has barely any basis to have learned that this factor should change the recommendation at all. I model that directly: the rule only recommends against the standard drug for bleeding-risk patients if the training set contained enough bleeding-risk cases to teach it that contraindication, a threshold this narrow training set doesn't reach.

In [ ]:
MIN_CASES_TO_LEARN_CONTRAINDICATION = 6


def derive_recommendation_rule(training_cases, min_cases_to_learn):
    """Returns whether the training set had enough bleeding-risk cases to learn the contraindication."""
    bleeding_risk_cases = [c for c in training_cases if c["has_bleeding_risk"]]
    return len(bleeding_risk_cases) >= min_cases_to_learn


def recommend_standard_drug(patient, learned_contraindication):
    """Returns True if the model recommends the standard, bleeding-risk drug for this patient."""
    if learned_contraindication and patient["has_bleeding_risk"]:
        return False
    return True


learned_contraindication = derive_recommendation_rule(training_cases, MIN_CASES_TO_LEARN_CONTRAINDICATION)
print("Bleeding-risk cases in the narrow training set:", len([c for c in training_cases if c["has_bleeding_risk"]]))
print("Did the model learn the bleeding-risk contraindication?", learned_contraindication)

for patient in deployment_patients:
    patient["recommended_standard_drug"] = recommend_standard_drug(patient, learned_contraindication)

unsafe_recommendations = [
    p for p in deployment_patients if p["has_bleeding_risk"] and p["recommended_standard_drug"]
]
print("Deployment patients with bleeding risk given an unsafe recommendation:", len(unsafe_recommendations))

## 6. Measuring a Second, Separate Problem: Drug Availability

Separately from safety, I check how many deployment patients were recommended a drug their own hospital can't actually give them, a mismatch that has nothing to do with any individual patient's risk factors and everything to do with which country's formulary the recommendation assumed.

In [ ]:
unavailable_recommendations = [
    p for p in deployment_patients if p["recommended_standard_drug"] and not p["drug_available"]
]
print("Deployment patients recommended a drug unavailable at their hospital:", len(unavailable_recommendations))
print("Out of total deployment patients:", len(deployment_patients))

## 7. Trying the Fix That Worked in an Earlier Notebook

In the facial recognition notebook, the fix for a group being underrepresented in training data was to train on genuinely comparable data instead. I try that here: rebuild the training set with a bleeding-risk rate that matches the real deployment population, and recheck both problems.

In [ ]:
better_training_cases = make_patients(N_TRAINING_CASES, DEPLOYMENT_BLEEDING_RISK_RATE, [HOSPITALS[0]])
learned_contraindication_better = derive_recommendation_rule(better_training_cases, MIN_CASES_TO_LEARN_CONTRAINDICATION)

for patient in deployment_patients:
    patient["recommended_standard_drug_better"] = recommend_standard_drug(patient, learned_contraindication_better)

unsafe_better = [
    p for p in deployment_patients if p["has_bleeding_risk"] and p["recommended_standard_drug_better"]
]
unavailable_better = [
    p for p in deployment_patients if p["recommended_standard_drug_better"] and not p["drug_available"]
]

print("Bleeding-risk cases in the improved training set:", len([c for c in better_training_cases if c["has_bleeding_risk"]]))
print("Unsafe recommendations with broader training data:", len(unsafe_better))
print("Unavailable-drug recommendations with broader training data:", len(unavailable_better))

## 8. Why Availability Still Isn't Fixed, and the Fix That Actually Closes It

Broader training data closes the safety gap, once the model has seen enough real bleeding-risk cases, it stops recommending the contraindicated drug to those patients, exactly like the facial recognition notebook. But the number of patients recommended a drug their hospital can't give them barely moves, because that problem was never about what the model learned, it's about a fact the model was never given at all: what's actually stocked and approved at the patient's own hospital. The fix for that has to sit outside the model entirely: an explicit check against each hospital's own formulary before any recommendation reaches a doctor.

In [ ]:
def apply_formulary_gate(patient):
    """Returns the final recommendation after checking it against the patient's own hospital formulary."""
    if patient["recommended_standard_drug_better"] and not patient["drug_available"]:
        return "flagged: not available locally, refer to alternative-treatment protocol"
    if patient["recommended_standard_drug_better"]:
        return "standard drug"
    return "alternative, bleeding-risk-safe protocol"


for patient in deployment_patients:
    patient["final_recommendation"] = apply_formulary_gate(patient)

still_unactionable = [
    p for p in deployment_patients
    if p["final_recommendation"] == "standard drug" and not p["drug_available"]
]
print("Patients still left with an unactionable recommendation after the formulary gate:", len(still_unactionable))

## 9. What This Matched in the Real Case

This two-part shape matches what actually happened. Independent reviews found Watson for Oncology's recommendations matched local oncologists closely for cancers like breast cancer, where treatment guidelines are unusually standardized worldwide, and matched far less often for cancers like rectal cancer, where legitimate practice genuinely varies more between institutions and countries, exactly where a narrow, single-institution training set would be expected to generalize worst.